In [1]:
# design_config helper

import json

design_config_json_path = 'example/scapCC4_NRD/design_config.json'

AAdict = {
    'Ala': 'A', 'Val': 'V', 'Met': 'M',
    'Phe': 'F', 'Tyr': 'Y', 'Gln': 'Q',
    'Thr': 'T', 'Gly': 'G', 'Leu': 'L',
    'Ile': 'I', 'Pro': 'P', 'Ser': 'S',
    'Cys': 'C', 'Trp': 'W', 'Asp': 'D',
    'Asn': 'N', 'Glu': 'E', 'Lys': 'K',
    'Arg': 'R', 'His': 'H'
}

AAs = list(AAdict.values())

design_config = {
    1:{
        'res_info': {
            19:'AILV',
            44:'AILV',
            111:'AILV',
        },
        'cavity':0,
        'tolerance': 20,
        'ligand_layer_vol':329.24/3
    },
    2:{
        'res_info': {
            16:'AILV',
            84:'AILV',
            114:'AILV'
        },
        'cavity':0,
        'tolerance': 20,
        'ligand_layer_vol':329.24/3
    },
    3:{
        'res_info': {
            12:'AILV',
            51:'AILV',
            118:'AILV'
        },
        'cavity':0,
        'tolerance': 20,
        'ligand_layer_vol':329.24/3
    }
    
}

with open(design_config_json_path, 'w')  as f:
    json.dump(design_config, f, indent=4)

In [2]:
# script arguments

from pathlib import Path

# most of this information should come in the form of script flags

# -o, --output_directory
output_directory = Path('./example/scapCC4_NRD')

# -r, --receptor_pdb_path
receptor_pdb_path = Path('./example/scapCC4_NRD/scapCC4.pdb')

# -l, --ligand_pdbqt_path
ligand_pdbqt_path = Path('./example/scapCC4_NRD/NileRed.pdbqt')

# -d, --desing_config_json_path
desing_config_json_path = design_config_json_path

# -g, --use_gradient_boosted_trees, default
use_gradient_boosted_trees = False

# -f, --faspr_path, default
faspr_path = Path('/opt/FASPR/FASPR')

# -n, --num_cpus, default
num_cpus = 16

# -s, --save_top_n, default
save_top_n = 3

# -c, --calc_seqs_only, default
calc_seqs_only = False

# -t, --timeout, default
timeout = 180

In [3]:
# local modules
from rasscol_src.rasscol_utils import *
from rasscol_src.general_utils import *

# builtin modules
import multiprocessing
import json, sys

rasscol = RASSCoL()

datestamp, timestamp = get_timestamp()

job_dir = output_directory 
job_dir.mkdir(exist_ok=True, parents=True)
config_json_path = job_dir / 'config.json'

with open(desing_config_json_path, 'r') as f:
    design_config = json.load(f)
    
starting_seq = pdb2seq(receptor_pdb_path)['A']

# Store all the design residue numbers as a list[int]
design_idx = [
    int(resnum)
    for layer in design_config
    for resnum in design_config[layer]['res_info'].keys()
]

for _layer in design_config:
    
    # Extract integer residue numbers for the current layer
    _layer_resnums = [int(_resnum) for _resnum in design_config[_layer]['res_info']]
    
    # Get the starting sequence at those residue positions
    _starting_seq_layer = [starting_seq[_resnum - 1] for _resnum in _layer_resnums]  # Adjust for 0-based indexing
    design_config[_layer]['starting_seq'] = _starting_seq_layer
    
    # Calculate the volume of the amino acids in the starting sequence
    _volume = sum(rasscol.aa_vol[_resname] for _resname in _starting_seq_layer)
    design_config[_layer]['volume'] = _volume
    
    # Calculate the target value
    design_config[_layer]['target'] = (_volume + design_config[_layer]['cavity']) - design_config[_layer]['ligand_layer_vol']

lig_length = get_mol_len(get_pdbqt_coords(ligand_pdbqt_path))

config = {
    'run':{
        'receptor_path':str(receptor_pdb_path),
        'ligand_path':str(ligand_pdbqt_path),
        'output_directory':str(job_dir),
        'FASPR_path':str(faspr_path),
        'num_cpus':num_cpus,
        'use_gradient_boosted_trees':use_gradient_boosted_trees,
        'save_top_n':save_top_n,
        'datestamp': datestamp,
        'timestamp': timestamp,
        'calc_seqs_only': calc_seqs_only,
        'cube_side_length': [lig_length*1.5]*3,
        'timeout':timeout
    },
    'versions':{
        'python':sys.version.split()[0],
        'vina':rasscol.vina_version 
    },
    'design':design_config
}

# write out config for logging
with config_json_path.open('w') as f:
    json.dump(config, f, indent=4)

scaffold_ca_coords = get_pdbqt_coords(receptor_pdb_path, ca_only=True)
pocket_ca_coords = [xyz for i, xyz in enumerate(scaffold_ca_coords, start=1) if i in design_idx]
config['run']['pocket_ca_centroid'] = get_centroid(pocket_ca_coords)
num_lig_atoms = len(get_pdbqt_coords(ligand_pdbqt_path))
docking_bock_size = get_mol_len(get_pdbqt_coords(ligand_pdbqt_path))*1.5

# Generate the sequence generators dictionary
seqs = {
    layer: rasscol.layer_sequence_generator(
        design_config[layer]['res_info'],
        design_config[layer]['target'],
        design_config[layer]['tolerance']
    ) for layer in design_config
}

# Define a wrapper function to run the sequence generator
def run_sequence_generator(starting_seq, seqs, design_idx, return_dict):
    return_dict["result"] = rasscol.sequence_generator(starting_seq, seqs, design_idx)

# Create a manager to handle shared data
manager = multiprocessing.Manager()
return_dict = manager.dict()

# Create the process
process = multiprocessing.Process(target=run_sequence_generator, args=(starting_seq, seqs, design_idx, return_dict))

# Start the process and set a timeout
process.start()
process.join(timeout=config['run']['timeout'])

# Check if the process is still alive
if process.is_alive():
    process.terminate()
    print("The function call timed out!")
else:
    seq_dict = return_dict["result"]

print(f'Sequences generated: {len(seq_dict)}')

if config['run']['calc_seqs_only']:
    pass

elif config['run']['use_gradient_boosted_trees']:
    pass

else:
    rasscol.run_parallel(starting_seq, design_idx, seq_dict, config['run']['pocket_ca_centroid'], num_lig_atoms, job_dir, receptor_pdb_path, ligand_pdbqt_path, config, design_config)
    
    results_csv = job_dir / 'RASSCoL_results.csv'
    
    # Read CSV data and store it in a list of dictionaries
    with results_csv.open() as f:
        reader = csv.DictReader(f)
        results = [row for row in reader]

    # Sort the results based on the 'vina_score_norm' column and get the top N
    data = sorted(results, key=lambda x: float(x['vina_score_norm']))

    # Write the list of dictionaries to a CSV file
    with open(results_csv, mode='w', newline='') as csvfile:
        # Create a csv.DictWriter object
        writer = csv.DictWriter(csvfile, fieldnames=data[0].keys())
        
        # Write the header (field names)
        writer.writeheader()
        
        # Write the data
        writer.writerows(data)
        
    rasscol.save_structures(config, job_dir / 'RASSCoL_results.csv')


Sequences generated: 729
Saved design 51 at example/scapCC4_NRD/51_NileRed.pdbqt
Saved design 48 at example/scapCC4_NRD/48_NileRed.pdbqt
Saved design 197 at example/scapCC4_NRD/197_NileRed.pdbqt
